# Antoine vapor-pressure correlation

**Prerequisite:** [Antoine vapor-pressure correlation](../../../docs/lecture-notes/thermodynamics/vapor-pressure/antoine/index.md)

This maintained laboratory evaluates an Antoine coefficient model with explicit coefficient units, validity range, and provenance. The selected coefficients are **synthetic instructional data**, not literature values or fitted material data.

## Learning objectives

1. Separate an Antoine coefficient model from its evaluator and result.
2. Preserve the coefficient table's native temperature and pressure units.
3. Evaluate temperatures supplied in kelvin and obtain pressures in pascals.
4. Check the represented formula independently for selected temperatures.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.thermal.thermo.vaporpressure.antoine import (
    AntoineTemperatureRange,
    AntoineVaporPressureModel,
)
from projectkoios.physkit.units import (
    PhysicalUnit,
    ScalarQuantity,
    Unitless,
    VectorQuantity,
)

In [ ]:
coefficient_temperature_unit = PhysicalUnit(expression="kelvin")
plot_pressure_unit = PhysicalUnit(expression="pascal")
valid_temperature_lower_kelvin = 400.0
valid_temperature_upper_kelvin = 800.0
temperature_sample_count = 201

model = AntoineVaporPressureModel(
    A=ScalarQuantity(magnitude=5.0, unit=Unitless()),
    B=ScalarQuantity(
        magnitude=1200.0,
        unit=coefficient_temperature_unit,
    ),
    C=ScalarQuantity(
        magnitude=-30.0,
        unit=coefficient_temperature_unit,
    ),
    coefficient_temperature_unit=coefficient_temperature_unit,
    coefficient_pressure_unit=PhysicalUnit(expression="bar"),
    valid_temperature_range=AntoineTemperatureRange(
        lower=ScalarQuantity(
            magnitude=valid_temperature_lower_kelvin,
            unit=coefficient_temperature_unit,
        ),
        upper=ScalarQuantity(
            magnitude=valid_temperature_upper_kelvin,
            unit=coefficient_temperature_unit,
        ),
    ),
    source="synthetic instructional coefficients",
)

temperature_values_kelvin = np.linspace(
    valid_temperature_lower_kelvin,
    valid_temperature_upper_kelvin,
    temperature_sample_count,
    dtype=np.float64,
)
temperatures = VectorQuantity(
    magnitude=temperature_values_kelvin,
    unit=PhysicalUnit(expression="kelvin"),
)
result = model.evaluate(
    temperatures=temperatures,
    pressure_unit=plot_pressure_unit,
)

In [ ]:
selected_indices = np.array([0, 100, 200], dtype=np.int64)
selected_temperatures = temperature_values_kelvin[selected_indices]
bar_to_pascal = 100_000.0
expected_pressures_pascal = (
    10.0
    ** (
        model.A.magnitude
        - model.B.magnitude / (selected_temperatures + model.C.magnitude)
    )
    * bar_to_pascal
)
pressure_relative_tolerance = 2.0e-15

assert np.allclose(
    result.pressures.magnitude[selected_indices],
    expected_pressures_pascal,
    rtol=pressure_relative_tolerance,
    atol=0.0,
)
assert np.all(np.diff(result.pressures.magnitude) > 0.0)

np.column_stack(
    (
        selected_temperatures,
        result.pressures.magnitude[selected_indices],
    )
)

In [ ]:
figure_size_inches = (8.0, 4.5)
figure, axis = plt.subplots(figsize=figure_size_inches)
axis.semilogy(
    result.temperatures.magnitude,
    result.pressures.magnitude,
    label="synthetic Antoine model",
)
axis.set(
    xlabel="temperature (K)",
    ylabel="vapor pressure (Pa)",
    title="Synthetic Antoine vapor-pressure example",
)
axis.grid(True)
axis.legend()
figure.tight_layout()
plt.show()

## Interpretation

The evaluator converts the pressure produced in the coefficient-native unit, bar, into pascals. The selected synthetic model produces increasing pressure throughout its declared interval. Range enforcement is an evaluation policy and can be disabled explicitly when inspecting extrapolation.

## Exercises

1. Define a synthetic coefficient set whose temperature convention is degrees Celsius and verify the Kelvin-to-Celsius conversion.
2. Change the coefficient-native pressure unit to Torr and compare equivalent parameterizations.
3. Evaluate exactly at both validity-range boundaries.
4. Choose coefficients that place $T_{\mathrm{native}}+C=0$ inside the input range and explain why evaluation rejects that point.
